# 02_RLVR_Part2_Training_v57.ipynb

### 🤖 Generation 16: Factor Registry Hygiene & De-biasing PPO Alpha Agent Training

- **State Space (46 Dimensions):**
  - `[0 : 12]` Cross-Sectional Factor Mean ($K=12$)
  - `[12 : 24]` Cross-Sectional Factor Std ($K=12$)
  - `[24 : 36]` Benchmark Factor Vector ($K=12$)
  - `[36 : 46]` Stationary Macro Vector (10 dims: `Mkt_Ret`, `Mkt_Ret_Z`, `Macro_Trend`, `Macro_Trend_Z`, `Yield_Curve_10Y2Y_Z`, `Macro_Trend_Vel_Z`, `Macro_Trend_Mom`, `Macro_Vix_Z`, `Macro_Vix_Ratio`, `Mkt_Vol_63d_Z`)

- **Action Space (16 Dimensions):**
  - `[0 : 12]` L2-Normalized Alpha Weights for 12 Orthogonal Factors
  - `[12]` Offset: Cross-sectional percentile start
  - `[13]` Width: Basket size $[5, 10]$
  - `[14]` Equity Exposure: Locked to 1.0 (Zero Cash Mandate)
  - `[15]` Active Tilt: Stock basket vs Benchmark Beta shelter $[0.10, 1.0]$

- **MTM Portfolio:** Overlapping $H = \text{config.holding\_period}$ sleeves with T+1 Close execution. Uniform intra-sleeve weighting ($1/K_t$).
- **Objective:** Maximize benchmark-relative Information Ratio and cumulative excess return over `config.benchmark` under 100% equity exposure.

In [1]:
# CELL 0: System Thread Locks, Cache Config & Reproducibility
import os
import torch
import multiprocessing

os.environ["CACHE_LOOKBACK"] = "41"
os.environ["CACHE_START_DATE"] = "1998-01-01"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
torch.set_num_threads(1)

print(f"CPU count: {multiprocessing.cpu_count()}")

CPU count: 8


In [2]:
# CELL 1: Start Execution Timer
import time

start_time = time.time()

In [3]:
# CELL 2: Environment & Path Setup
import sys
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root_default = Path(
        r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2"
    )
    rl_root = rl_root_default if rl_root_default.exists() else Path.cwd()

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

Mounted at /content/drive


In [4]:
# AUTOMATED TRAINING GATE: Run before training starts
import pytest

ret_code = pytest.main(["-x", "tests/test_mtm_invariants.py", "-q"])
if ret_code != 0:
    raise SystemError(
        "❌ TRAINING ABORTED: MTM Invariant tripwires failed! Check tests/test_mtm_invariants.py"
    )
print("✅ ALL MTM INVARIANT TRIPWIRES PASSED. Safe to proceed with training.")

...                                                                      [100%]
3 passed in 16.35s
✅ ALL MTM INVARIANT TRIPWIRES PASSED. Safe to proceed with training.


In [5]:
# CELL 3: Imports
import copy
import gc
import json
import random
from typing import Any, Dict, Optional

import gymnasium as gym
import numpy as np
import pandas as pd

from core.paths import OUTPUT_DIR, LOCAL_DATA_DIR
from core.settings import CacheConfig, TradingConfig
from core.quant import QuantUtils
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from rl_discovery.adapter import RLVRGymEnv
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.trainer import PPOTrainer, RolloutBuffer
from rl_discovery.validator import AgentEvaluator

In [6]:
# CELL 4: Load AlphaCache, Return Matrices & Split Timeline
print("📦 Loading preprocessed market data & AlphaCache...")
data = load_processed_data()
df_ohlcv = data.df_ohlcv
macro_df = data.macro_df
features_df = data.features_df
config = TradingConfig()
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

# 1. Load and Validate AlphaCache (Feature Cube)
cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()
print(f"AlphaCache path: {cache_file_path}")
if not cache_file_path.exists():
    raise FileNotFoundError(
        f"❌ AlphaCache file missing at {cache_file_path}! Run Notebook 01 first."
    )

feature_cube = pd.read_parquet(cache_file_path)

# Gen 16 Invariant Check: Feature cube must contain exactly K=12 orthogonal factors
k_factors = feature_cube.shape[1]
assert (
    k_factors == 12
), f"❌ AlphaCache factor count mismatch! Expected 12 factors, found {k_factors}: {feature_cube.columns.tolist()}"
print(
    f"✅ AlphaCache validated: {k_factors} orthogonal factors verified across {len(feature_cube)} rows."
)

# 2. Stationary Macro Vector Validation (Locked to 10 Dimensions)
final_10_cols = [
    "Mkt_Ret",
    "Mkt_Ret_Z",
    "Macro_Trend",
    "Macro_Trend_Z",
    "Yield_Curve_10Y2Y_Z",
    "Macro_Trend_Vel_Z",
    "Macro_Trend_Mom",
    "Macro_Vix_Z",
    "Macro_Vix_Ratio",
    "Mkt_Vol_63d_Z",
]
missing_macro = [c for c in final_10_cols if c not in macro_df.columns]
assert (
    not missing_macro
), f"❌ macro_df is missing required stationary columns: {missing_macro}"
macro_df = macro_df[final_10_cols]
assert macro_df.shape[1] == 10, f"❌ Expected 10 macro columns, got {macro_df.shape[1]}"
print("✅ Macro dataset validated: Exactly 10 stationary columns confirmed.")

# 3. Forward 1-Day MTM Simple Return Matrix (P_{t+1} - P_t) / P_t
simple_ret_matrix = df_close.pct_change(1, fill_method=None).shift(-1)
simple_ret_matrix["CASH"] = 0.0

# 4. Timeline Synchronization & Non-Leaking Splits
valid_dates = feature_cube.index.get_level_values("Date").unique()
valid_calendar = trading_calendar[trading_calendar.isin(valid_dates)]

VAL_START = pd.Timestamp("2016-01-01")
TEST_START = pd.Timestamp("2022-04-01")

cal_train_raw = valid_calendar[valid_calendar < VAL_START]
cal_val_raw = valid_calendar[
    (valid_calendar >= VAL_START) & (valid_calendar < TEST_START)
]
cal_test_raw = valid_calendar[valid_calendar >= TEST_START]

# Purge holding period leakage buffer
hp = config.holding_period
cal_train = cal_train_raw[:-hp] if len(cal_train_raw) > hp else cal_train_raw
cal_val = cal_val_raw[:-hp] if len(cal_val_raw) > hp else cal_val_raw
cal_test = cal_test_raw

print(f"✅ Data splits ready with {hp}-day leakage buffer dropped:")
print(
    f"   • Train: {cal_train.min().strftime('%Y-%m-%d')} to {cal_train.max().strftime('%Y-%m-%d')} ({len(cal_train)} days)"
)
print(
    f"   • Val  : {cal_val.min().strftime('%Y-%m-%d')} to {cal_val.max().strftime('%Y-%m-%d')} ({len(cal_val)} days)"
)
print(
    f"   • Test : {cal_test.min().strftime('%Y-%m-%d')} to {cal_test.max().strftime('%Y-%m-%d')} ({len(cal_test)} days)"
)

del df_ohlcv
gc.collect()

📦 Loading preprocessed market data & AlphaCache...
AlphaCache path: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data/alpha_cache_41d_1998.parquet
✅ AlphaCache validated: 12 orthogonal factors verified across 5035126 rows.
✅ Macro dataset validated: Exactly 10 stationary columns confirmed.
✅ Data splits ready with 5-day leakage buffer dropped:
   • Train: 1998-01-02 to 2015-12-23 (4524 days)
   • Val  : 2016-01-04 to 2022-03-24 (1568 days)
   • Test : 2022-04-01 to 2026-09-04 (1111 days)


14

In [7]:
# CELL 8: Hyperparameter Grid Configuration — Generation 16 (Factor Registry Hygiene & De-biasing)
# Causal Hypothesis: Decoupling the factor registry from collinear momentum/volatility metrics
# (reducing condition number kappa from ~45 to < 10) allows PPO gradients to load directly onto
# genuine idiosyncratic alpha compounders without synthetic curve-flattening arbitrage.
# Baseline Control: Run 217 Ridge (Scale=25, Gamma=0.88, LossAv=0.35, Up=1.18, Tilt=0.10)
# Invariants: WARMUP_EPOCHS = 25, PATIENCE = 30, MIN_EPOCHS = 70, NUM_EPOCHS = 90, Uniform Weighting (1/K_t)

SLICE_NUM = 2  # Set to 1, 2, or 3 for distributed training

param_grid_sweep = [
    # ---------------- SLICE 1: DIRECT CONTROL REPLICATION & MULTI-SEED (RUN 217 RIDGE) ----------------
    {
        "ID": "P16_RegHygiene_Scale25_Gamma88_LossAv035_Up118_Tilt10_Width5",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 1.18,
        "loss_aversion_penalty": 0.35,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.10,
        "min_width": 5,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [42],  # Direct causal control against Run 217
        "NUM_EPOCHS": 90,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 30,
        "WARMUP_EPOCHS": 25,
        "MIN_EPOCHS": 70,
        "NUM_STEPS": 512,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 5,
    },
    {
        "ID": "P16_RegHygiene_Scale25_Gamma88_LossAv035_Up118_Tilt10_Width5",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 1.18,
        "loss_aversion_penalty": 0.35,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.10,
        "min_width": 5,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [101],  # Multi-seed validation
        "NUM_EPOCHS": 90,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 30,
        "WARMUP_EPOCHS": 25,
        "MIN_EPOCHS": 70,
        "NUM_STEPS": 512,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 5,
    },
    {
        "ID": "P16_RegHygiene_Scale25_Gamma88_LossAv035_Up118_Tilt10_Width5",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 1.18,
        "loss_aversion_penalty": 0.35,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.10,
        "min_width": 5,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [777],  # Multi-seed validation
        "NUM_EPOCHS": 90,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 30,
        "WARMUP_EPOCHS": 25,
        "MIN_EPOCHS": 70,
        "NUM_STEPS": 512,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 5,
    },
    # ---------------- SLICE 2: BENCHMARK SHELTER & ASYMMETRY SWEET-SPOT (SEED 42) ----------------
    {
        "ID": "P16_RegHygiene_Scale25_Gamma88_LossAv035_Up118_Tilt08_Width5",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 1.18,
        "loss_aversion_penalty": 0.35,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.08,  # Test deeper 92% benchmark shelter
        "min_width": 5,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 90,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 30,
        "WARMUP_EPOCHS": 25,
        "MIN_EPOCHS": 70,
        "NUM_STEPS": 512,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 5,
    },
    {
        "ID": "P16_RegHygiene_Scale25_Gamma88_LossAv030_Up118_Tilt10_Width5",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 1.18,
        "loss_aversion_penalty": 0.30,  # Test lower loss aversion with orthogonal crash filters
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.10,
        "min_width": 5,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 90,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 30,
        "WARMUP_EPOCHS": 25,
        "MIN_EPOCHS": 70,
        "NUM_STEPS": 512,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 5,
    },
    {
        "ID": "P16_RegHygiene_Scale25_Gamma88_LossAv035_Up120_Tilt10_Width5",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 1.20,  # Re-test Up 1.20 exploration boundary
        "loss_aversion_penalty": 0.35,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.10,
        "min_width": 5,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [42],
        "NUM_EPOCHS": 90,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 30,
        "WARMUP_EPOCHS": 25,
        "MIN_EPOCHS": 70,
        "NUM_STEPS": 512,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 5,
    },
    # ---------------- SLICE 3: MULTI-SEED REPLICATION (DEEP SHELTER & LOSS-AVERSION) ----------------
    {
        "ID": "P16_RegHygiene_Scale25_Gamma88_LossAv035_Up118_Tilt08_Width5",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 1.18,
        "loss_aversion_penalty": 0.35,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.08,
        "min_width": 5,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [101],
        "NUM_EPOCHS": 90,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 30,
        "WARMUP_EPOCHS": 25,
        "MIN_EPOCHS": 70,
        "NUM_STEPS": 512,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 5,
    },
    {
        "ID": "P16_RegHygiene_Scale25_Gamma88_LossAv030_Up118_Tilt10_Width5",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 1.18,
        "loss_aversion_penalty": 0.30,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.10,
        "min_width": 5,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [101],
        "NUM_EPOCHS": 90,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 30,
        "WARMUP_EPOCHS": 25,
        "MIN_EPOCHS": 70,
        "NUM_STEPS": 512,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 5,
    },
    {
        "ID": "P16_RegHygiene_Scale25_Gamma88_LossAv035_Up118_Tilt08_Width5",
        "LR": 2.0e-4,
        "CRITIC_LR": 8.0e-4,
        "ENT_COEF_START": 0.0050,
        "ENT_COEF_END": 0.0005,
        "UPSIDE_ALPHA_MULT": 1.18,
        "loss_aversion_penalty": 0.35,
        "max_cash_pct": 0.0,
        "min_active_tilt": 0.08,
        "min_width": 5,
        "GAMMA": 0.88,
        "GAE_LAMBDA": 0.95,
        "REWARD_SCALE": 25.0,
        "SEEDS": [777],
        "NUM_EPOCHS": 90,
        "UPDATE_EPOCHS": 4,
        "PATIENCE": 30,
        "WARMUP_EPOCHS": 25,
        "MIN_EPOCHS": 70,
        "NUM_STEPS": 512,
        "MINI_BATCH_SIZE": 256,
        "NUM_ENVS": 8,
        "EVAL_FREQ": 2,
        "ROLLING_EVAL_WINDOW": 5,
    },
]

slice_map = {
    1: param_grid_sweep[0:3],
    2: param_grid_sweep[3:6],
    3: param_grid_sweep[6:9],
}

if SLICE_NUM not in slice_map:
    raise ValueError(f"❌ SLICE_NUM must be 1, 2, or 3. Got: {SLICE_NUM}")

grid = slice_map[SLICE_NUM]
print("=" * 75)
print(
    f"🔥 ACTIVE TRAINING SESSION: GENERATION 16 (SLICE {SLICE_NUM} of 3, {len(grid)} Configurations)"
)
print("=" * 75)
for i, p in enumerate(grid, 1):
    print(
        f"  [{i}/{len(grid)}] ID: {p['ID']:<60} | Seed: {p['SEEDS'][0]} | "
        f"Up: {p['UPSIDE_ALPHA_MULT']} | Tilt: {p['min_active_tilt']} | LossAv: {p['loss_aversion_penalty']}"
    )
print("=" * 75)

🔥 ACTIVE TRAINING SESSION: GENERATION 16 (SLICE 2 of 3, 3 Configurations)
  [1/3] ID: P16_RegHygiene_Scale25_Gamma88_LossAv035_Up118_Tilt08_Width5 | Seed: 42 | Up: 1.18 | Tilt: 0.08 | LossAv: 0.35
  [2/3] ID: P16_RegHygiene_Scale25_Gamma88_LossAv030_Up118_Tilt10_Width5 | Seed: 42 | Up: 1.18 | Tilt: 0.1 | LossAv: 0.3
  [3/3] ID: P16_RegHygiene_Scale25_Gamma88_LossAv035_Up120_Tilt10_Width5 | Seed: 42 | Up: 1.2 | Tilt: 0.1 | LossAv: 0.35


In [8]:
# CELL 6: Environment Factory
def create_fresh_environments(
    base_config: TradingConfig,
    grid_params: Dict[str, Any],
    df_close: pd.DataFrame,
    macro_df: pd.DataFrame,
    feature_cube: pd.DataFrame,
    cal_train: pd.DatetimeIndex,
    cal_val: pd.DatetimeIndex,
    cal_test: pd.DatetimeIndex,
):
    run_config = copy.deepcopy(base_config)

    if "loss_aversion_penalty" in grid_params:
        run_config.loss_aversion_penalty = float(grid_params["loss_aversion_penalty"])
    if "UPSIDE_ALPHA_MULT" in grid_params:
        run_config.upside_alpha_mult = float(grid_params["UPSIDE_ALPHA_MULT"])
    if "max_cash_pct" in grid_params:
        run_config.max_cash_pct = float(grid_params["max_cash_pct"])
    if "min_active_tilt" in grid_params:
        run_config.min_active_tilt = float(grid_params["min_active_tilt"])
    if "min_width" in grid_params:
        run_config.min_basket_width = int(grid_params["min_width"])
    elif "min_basket_width" in grid_params:
        run_config.min_basket_width = int(grid_params["min_basket_width"])

    simple_rets = df_close.pct_change(1, fill_method=None).shift(-1)
    simple_rets["CASH"] = 0.0

    def make_train_env():
        def thunk():
            e = DiscoveryEnv(
                feature_cube=feature_cube,
                simple_ret_matrix=simple_rets,
                calendar=cal_train,
                macro_df=macro_df,
                config=run_config,
                randomize_start=True,
                episode_steps=grid_params["NUM_STEPS"],
            )
            return RLVRGymEnv(e, macro_df)

        return thunk

    env_val = DiscoveryEnv(
        feature_cube=feature_cube,
        simple_ret_matrix=simple_rets,
        calendar=cal_val,
        macro_df=macro_df,
        config=run_config,
        randomize_start=False,
        episode_steps=0,  # Strict full-calendar validation (no 512-step cutoff)
    )
    gym_val = RLVRGymEnv(env_val, macro_df)
    gym_val.is_training = False

    env_test = DiscoveryEnv(
        feature_cube=feature_cube,
        simple_ret_matrix=simple_rets,
        calendar=cal_test,
        macro_df=macro_df,
        config=run_config,
        randomize_start=False,
        episode_steps=0,  # Strict full-calendar test bench
    )
    gym_test = RLVRGymEnv(env_test, macro_df)
    gym_test.is_training = False

    return make_train_env, gym_val, gym_test, run_config

In [9]:
# ==============================================================================
# CELL 10: Training Engine with Generation 11 Composite Fitness Early Stopping
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True, warn_only=True)


def get_hp_tag(params: Dict[str, Any], seed: int = 42) -> str:
    cid = params.get("ID", f"run_s{seed}")
    return f"{cid}_s{seed}"


def update_model_catalog(catalog_path: Path, record: Dict[str, Any]):
    """Atomically append run parameters and performance metrics to model_catalog.parquet."""
    df_new = pd.DataFrame([record])
    if catalog_path.exists():
        df_existing = pd.read_parquet(catalog_path)
        df_catalog = (
            pd.concat([df_existing, df_new], ignore_index=True)
            .drop_duplicates(subset=["checkpoint_name"], keep="last")
            .reset_index(drop=True)
        )
    else:
        df_catalog = df_new

    df_catalog.to_parquet(catalog_path, index=False)


def train_agent_for_grid(
    grid_params: Dict[str, Any],
    make_train_env,
    gym_val: RLVRGymEnv,
    checkpoint_dir: Path,
    run_config: Optional[TradingConfig] = None,
):
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    cfg = run_config or TradingConfig()

    seeds = grid_params["SEEDS"]
    num_epochs = grid_params["NUM_EPOCHS"]
    warmup_epochs = grid_params.get("WARMUP_EPOCHS", 25)
    min_epochs = grid_params.get("MIN_EPOCHS", 70)
    num_envs = grid_params["NUM_ENVS"]

    grid_best_metric = -np.inf
    grid_best_model_path: Optional[Path] = None
    grid_best_seed = None
    grid_best_history = []
    grid_best_state_dict = None

    for seed in seeds:
        print(f"\n   -> 🚀 SEED: {seed}")
        set_seed(seed)
        hp_tag = get_hp_tag(grid_params, seed)

        # Clean lingering checkpoints for this specific hp_tag
        for existing_file in checkpoint_dir.glob(f"model_{hp_tag}_*.pt"):
            try:
                existing_file.unlink()
            except OSError:
                pass

        if "google.colab" in sys.modules:
            envs = gym.vector.AsyncVectorEnv(
                [make_train_env() for _ in range(num_envs)]
            )
        else:
            envs = gym.vector.SyncVectorEnv([make_train_env() for _ in range(num_envs)])

        obs_shape = envs.single_observation_space.shape
        action_shape = envs.single_action_space.shape
        assert obs_shape is not None and action_shape is not None

        obs_dim = int(obs_shape[0])
        action_dim = int(action_shape[0])

        # ---> GENERATION 16 CONTRACT TRIPWIRES <---
        assert (
            obs_dim == 46
        ), f"❌ State dimension shattered! Expected 46 (3*12 + 10), got {obs_dim}"
        assert (
            action_dim == 16
        ), f"❌ Action dimension shattered! Expected 16 (12 + 4), got {action_dim}"

        agent = AbsoluteZeroAgent(obs_dim=obs_dim, action_dim=action_dim).to(device)
        trainer = PPOTrainer(
            agent,
            lr=grid_params["LR"],
            critic_lr=grid_params.get("CRITIC_LR", grid_params["LR"] * 4.0),
            clip_coef=0.2,
            clip_vloss=True,
            ent_coef=grid_params.get("ENT_COEF_START", 0.0050),
        )

        buffer = RolloutBuffer(
            num_steps=grid_params["NUM_STEPS"],
            num_envs=num_envs,
            obs_dim=obs_dim,
            action_dim=action_dim,
            device=device,
            gamma=grid_params.get("GAMMA", cfg.gamma),
            gae_lambda=grid_params.get("GAE_LAMBDA", cfg.gae_lambda),
        )

        seed_best_rolling_metric = -np.inf
        seed_current_checkpoint_path: Optional[Path] = None
        patience_counter = 0
        seed_history = []
        val_metric_history = []
        rolling_window = grid_params.get("ROLLING_EVAL_WINDOW", 5)

        for epoch in range(num_epochs):
            obs, _ = envs.reset(seed=seed) if epoch == 0 else envs.reset()
            epoch_rewards = []

            for step in range(grid_params["NUM_STEPS"]):
                obs_tensor = torch.tensor(obs, dtype=torch.float32).to(device)
                with torch.no_grad():
                    action, logprob, _, value = agent.get_action_and_value(obs_tensor)

                raw_action = action.cpu().numpy()
                next_obs, rewards, terminations, truncations, _ = envs.step(raw_action)

                if np.isnan(rewards).any() or np.isinf(rewards).any():
                    raise FloatingPointError(
                        f"NaN/Inf reward detected at Epoch {epoch+1}, Step {step}"
                    )

                reward_scale = grid_params.get("REWARD_SCALE", 25.0)
                scaled_rewards = rewards * reward_scale

                buffer.add(
                    obs=obs,
                    action=action,
                    logprob=logprob,
                    reward=scaled_rewards,
                    value=value,
                    terminations=terminations,
                    truncations=truncations,
                )
                epoch_rewards.append(rewards.mean())
                obs = next_obs

            # Bootstrap V(s_T)
            obs_tensor = torch.tensor(obs, dtype=torch.float32).to(device)
            next_term_tensor = torch.tensor(terminations, dtype=torch.float32).to(
                device
            )
            with torch.no_grad():
                next_values = agent.get_value(obs_tensor)

            buffer.compute_advantages(next_values, next_termination=next_term_tensor)

            trainer.update_schedules(
                epoch + 1,
                num_epochs,
                ent_start=grid_params.get("ENT_COEF_START", 0.0050),
                ent_end=grid_params.get("ENT_COEF_END", 0.0005),
            )

            diagnostics = trainer.update(
                buffer,
                update_epochs=grid_params["UPDATE_EPOCHS"],
                mini_batch_size=grid_params["MINI_BATCH_SIZE"],
            )
            buffer.step = 0

            diagnostics["epoch"] = epoch + 1
            diagnostics["avg_reward"] = float(np.mean(epoch_rewards))
            seed_history.append(diagnostics)

            if (epoch + 1) % grid_params["EVAL_FREQ"] == 0:
                train_scaler = envs.get_attr("scaler")[0]
                gym_val.scaler.load_state(train_scaler)

                val_results = AgentEvaluator.evaluate(agent, gym_val, device)

                # ==========================================================
                # GENERATION 12 CONTRACT: BENCHMARK-RELATIVE ACTIVE ALPHA FITNESS
                # Objective: Maximize Information Ratio & Cumulative Spread.
                # Invariant: Low-beta benchmark huggers with negative alpha
                #            receive a strict penalty (Fitness < 0).
                # ==========================================================
                val_sharpe = float(val_results["sharpe_ratio"])
                val_ir = float(val_results["information_ratio"])
                val_excess = float(val_results["excess_return"])

                # Single source of truth from QuantUtils kernel
                val_fitness = QuantUtils.compute_composite_fitness(
                    excess_return=val_excess,
                    information_ratio=val_ir,
                )
                val_metric_history.append(val_fitness)

                rolling_fitness = float(np.mean(val_metric_history[-rolling_window:]))
                is_warmup = (epoch + 1) <= warmup_epochs
                status_icon = "🔥 [WARMUP]" if is_warmup else ""

                print(
                    f"      Ep {epoch+1:03d} | Rew: {diagnostics['avg_reward']:.4f} | "
                    f"Loss: {diagnostics['total_loss']:.4f} | EV: {diagnostics['explained_variance']:.3f} | "
                    f"Fit: {val_fitness:.4f} | Roll-{rolling_window} Fit: {rolling_fitness:.4f} | "
                    f"Val IR: {val_ir:.3f} | Excess: {val_excess*100:+.2f}% | Val Sh: {val_sharpe:.3f} {status_icon}"
                )

                if not is_warmup:
                    if rolling_fitness > seed_best_rolling_metric:
                        seed_best_rolling_metric = rolling_fitness
                        patience_counter = 0

                        new_checkpoint_name = f"model_{hp_tag}_ep_{epoch+1}.pt"
                        new_checkpoint_path = checkpoint_dir / new_checkpoint_name

                        scaler_state = {
                            "mean": train_scaler.mean.copy(),
                            "var": train_scaler.var.copy(),
                            "count": train_scaler.count,
                        }

                        checkpoint_payload = {
                            "model_state_dict": agent.state_dict(),
                            "scaler_state": scaler_state,
                            "val_fitness": val_fitness,
                            "rolling_val_fitness": rolling_fitness,
                            "val_sharpe": val_sharpe,
                            "val_sortino": val_results["sortino_ratio"],
                            "val_excess_return": val_excess,
                            "val_info_ratio": val_ir,
                            "epoch": epoch + 1,
                            "seed": seed,
                            "grid_params": grid_params,
                            "holding_period": cfg.holding_period,
                            "benchmark": cfg.benchmark,
                            "num_steps": grid_params["NUM_STEPS"],
                            "num_envs": num_envs,
                            "total_timesteps_per_epoch": grid_params["NUM_STEPS"]
                            * num_envs,
                            "training_history": list(seed_history),
                        }

                        torch.save(checkpoint_payload, str(new_checkpoint_path))

                        catalog_record = {
                            "checkpoint_name": new_checkpoint_name,
                            "model_path": str(new_checkpoint_path),
                            "epoch": epoch + 1,
                            "seed": seed,
                            "val_fitness": val_fitness,
                            "rolling_val_fitness": rolling_fitness,
                            "val_sharpe": val_sharpe,
                            "val_info_ratio": val_ir,
                            "val_excess_return": val_excess,
                            "num_steps": grid_params["NUM_STEPS"],
                            "num_envs": num_envs,
                            "holding_period": cfg.holding_period,
                            "benchmark": cfg.benchmark,
                            "min_active_tilt": grid_params.get("min_active_tilt"),
                            "min_width": grid_params.get("min_width"),
                            "lr": grid_params["LR"],
                            "gamma": grid_params.get("GAMMA", cfg.gamma),
                            "reward_scale": grid_params.get("REWARD_SCALE", 25.0),
                            "timestamp": pd.Timestamp.now().isoformat(),
                        }
                        update_model_catalog(
                            checkpoint_dir.parent / "model_catalog.parquet",
                            catalog_record,
                        )

                        # Clean up previous local checkpoint to conserve disk space
                        if (
                            seed_current_checkpoint_path is not None
                            and seed_current_checkpoint_path.exists()
                        ):
                            try:
                                seed_current_checkpoint_path.unlink()
                            except OSError:
                                pass

                        seed_current_checkpoint_path = new_checkpoint_path
                        print(
                            f"         🟢 NEW BEST! Retained model (Roll IR-Fit: {rolling_fitness:.4f} | IR: {val_ir:.3f} | Excess: {val_excess*100:+.2f}%)"
                        )

                        if rolling_fitness > grid_best_metric:
                            grid_best_metric = rolling_fitness
                            grid_best_model_path = new_checkpoint_path
                            grid_best_seed = seed
                            grid_best_history = list(seed_history)
                            grid_best_state_dict = copy.deepcopy(agent.state_dict())
                    else:
                        patience_counter += 1
                        print(
                            f"         🔴 No improvement. Patience: {patience_counter}/{grid_params['PATIENCE']}"
                        )

                    # Generation 12 Invariant: Enforce MIN_EPOCHS before early stopping triggers
                    if (epoch + 1) >= min_epochs and patience_counter >= grid_params[
                        "PATIENCE"
                    ]:
                        print(
                            f"      🛑 EARLY STOPPING triggered for Seed {seed} at Epoch {epoch+1}"
                        )
                        break

        envs.close()
        del agent, trainer, buffer, envs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return (
        grid_best_model_path,
        grid_best_metric,
        grid_best_seed,
        grid_best_history,
        grid_best_state_dict,
    )

In [10]:
# ==============================================================================
# CELL 11: Grid Execution Loop (Generation 11)
# ==============================================================================
checkpoint_dir = OUTPUT_DIR / "model_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

OVERWRITE_EXISTING = True

print("\n" + "=" * 60)
print(f"🏆 STARTING GENERATION 16 FACTOR HYGIENE SWEEP (SLICE {SLICE_NUM}) 🏆")
print("=" * 60)

for idx, params in enumerate(grid):
    current_seed = params["SEEDS"][0]
    hp_tag = get_hp_tag(params, current_seed)

    done_marker = checkpoint_dir / f".completed_{hp_tag}"
    if done_marker.exists() and not OVERWRITE_EXISTING:
        print(
            f"\n✅ [SKIPPING] GRID COMBO {idx+1}/{len(grid)} - Already completed: {hp_tag}"
        )
        continue
    elif done_marker.exists() and OVERWRITE_EXISTING:
        try:
            done_marker.unlink()
        except OSError:
            pass

    print(f"\n\n{'='*50}")
    print(f"🔧 RUNNING GRID COMBO {idx+1}/{len(grid)}: {params['ID']}")
    print(json.dumps({k: v for k, v in params.items() if k != "SEEDS"}, indent=2))
    print(f"{'='*50}")

    make_train_env, gym_val, gym_test, run_config = create_fresh_environments(
        base_config=config,
        grid_params=params,
        df_close=df_close,
        macro_df=macro_df,
        feature_cube=feature_cube,
        cal_train=cal_train,
        cal_val=cal_val,
        cal_test=cal_test,
    )

    best_model_path, best_fitness, best_seed, best_history, best_state_dict = (
        train_agent_for_grid(
            grid_params=params,
            make_train_env=make_train_env,
            gym_val=gym_val,
            checkpoint_dir=checkpoint_dir,
            run_config=run_config,
        )
    )

    done_marker.touch()

    if best_state_dict is None:
        print("⚠️ No model survived warmup for this configuration.")
        continue

    champion_name = best_model_path.name if best_model_path else "None"
    print(
        f"\n⭐ [GRID {idx+1} COMPLETE] Best Val Fitness: {best_fitness:.3f} | Best Model Saved: {champion_name}"
    )


🏆 STARTING GENERATION 16 FACTOR HYGIENE SWEEP (SLICE 2) 🏆


🔧 RUNNING GRID COMBO 1/3: P16_RegHygiene_Scale25_Gamma88_LossAv035_Up118_Tilt08_Width5
{
  "ID": "P16_RegHygiene_Scale25_Gamma88_LossAv035_Up118_Tilt08_Width5",
  "LR": 0.0002,
  "CRITIC_LR": 0.0008,
  "ENT_COEF_START": 0.005,
  "ENT_COEF_END": 0.0005,
  "UPSIDE_ALPHA_MULT": 1.18,
  "loss_aversion_penalty": 0.35,
  "max_cash_pct": 0.0,
  "min_active_tilt": 0.08,
  "min_width": 5,
  "GAMMA": 0.88,
  "GAE_LAMBDA": 0.95,
  "REWARD_SCALE": 25.0,
  "NUM_EPOCHS": 90,
  "UPDATE_EPOCHS": 4,
  "PATIENCE": 30,
  "WARMUP_EPOCHS": 25,
  "MIN_EPOCHS": 70,
  "NUM_STEPS": 512,
  "MINI_BATCH_SIZE": 256,
  "NUM_ENVS": 8,
  "EVAL_FREQ": 2,
  "ROLLING_EVAL_WINDOW": 5
}

   -> 🚀 SEED: 42
      Ep 002 | Rew: -0.0002 | Loss: -0.0744 | EV: -1.178 | Fit: -7.3004 | Roll-5 Fit: -7.3004 | Val IR: -0.703 | Excess: -36.50% | Val Sh: 0.736 🔥 [WARMUP]
      Ep 004 | Rew: -0.0002 | Loss: -0.0849 | EV: -0.163 | Fit: -4.5473 | Roll-5 Fit: -5.9238 | Val IR: -0

In [11]:
# CELL 9: Execution Timing Summary
end_time = time.time()
print("\n" + "=" * 60)
print(
    f"🏆 TRAINING COMPLETED! Total Duration: {time.strftime('%H:%M:%S', time.gmtime(end_time - start_time))}"
)
print("Run Notebook 03a to evaluate the saved models Out-of-Sample.")
print("=" * 60)


🏆 TRAINING COMPLETED! Total Duration: 00:31:16
Run Notebook 03a to evaluate the saved models Out-of-Sample.


In [12]:
# CELL 10: Runtime Cleanup / Disconnect (Colab Safe)
if RUNNING_IN_COLAB:
    from google.colab import runtime  # type: ignore

    runtime.unassign()
    print("Runtime unassigned. Session disconnected.")

Runtime unassigned. Session disconnected.
